### PART1 VectorDB : 상품의미검색
- 상품설명을 임베딩해서 맥락 기반의 검색 수행
### PART2 GraphDB : 구매/리뷰 
- 유저-상품-리뷰간의 그래프생성 및 분석
### PART3 GraphRAG & 추천
- 의미검색결과에서 신뢰도(그래프 중심성)높은 상품 필터링

In [1]:
import chromadb
from chromadb.utils import embedding_functions
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

products = [
    {"id": "p1", "name": "프로그래밍용 고성능 노트북", "desc": "최신 14세대 CPU와 32GB RAM을 탑재하여 무거운 딥러닝 및 개발 작업에 최적화된 랩탑입니다.", "category": "가전"},
    {"id": "p2", "name": "초경량 사무용 랩탑", "desc": "1kg 미만의 가벼운 무게로 카페나 도서관에서 문서 작업 및 웹서핑을 하기에 매우 좋은 노트북.", "category": "가전"},
    {"id": "p3", "name": "전문가용 미러리스 카메라", "desc": "4K 60fps 동영상 촬영과 빠르고 정확한 AF를 지원하여 유튜버 및 프로 사진작가에게 적합한 카메라.", "category": "가전"},
    {"id": "p4", "name": "편안한 러닝화", "desc": "쿠셔닝이 뛰어나고 통풍이 잘 되어 장시간 달리기나 헬스장 운동 시 발의 피로를 최소화해주는 운동화.", "category": "패션"},
    {"id": "p5", "name": "방수 트레킹화", "desc": "고어텍스 소재로 비오는 날이나 거친 산악 지형에서도 발을 쾌적하게 보호하는 등산용 신발.", "category": "패션"}
]

In [3]:
# 1. Chroma 인메모리 클라이언트 및 다국어 임베딩 설정
client = chromadb.Client()
st_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='paraphrase-multilingual-MiniLM-L12-v2'
)
# 기존 컬렉션 초기화 후 재생성
try:
    client.delete_collection('ecommerce')
except Exception:
    pass    
product_collection =  client.create_collection(
    name ='ecommerce',
    embedding_function=st_ef,
    metadata={'hnsw:space':'cosine'}
)
# 2. 가상의 이커머스 상품 데이터 (노트북, 카메라, 신발 등)
# 3. Vector DB에 데이터 삽입
product_collection.add(
    ids = [f'product_{i:03d}' for i, p in enumerate(products)],
    documents=[p['desc'] for p in products],
    metadatas=[
        { 'name':p['name'],'category':p['category']}
            for p in products        
    ]
)
# 4. 의미 기반 검색 테스트
query = '산에 갈 때 신기 좋은 튼튼한 신발'
results = product_collection.query(
    query_texts=[query],
    n_results=2  # top_2
)
for doc, meta,dist in zip(results['documents'][0],results['metadatas'][0],results['distances'][0]):
    similarity = 1- dist # 코사인거리->유사도 변환    
    print(similarity,doc,meta,dist)

0.785952091217041 고어텍스 소재로 비오는 날이나 거친 산악 지형에서도 발을 쾌적하게 보호하는 등산용 신발. {'category': '패션', 'name': '방수 트레킹화'} 0.21404790878295898
0.4105861186981201 쿠셔닝이 뛰어나고 통풍이 잘 되어 장시간 달리기나 헬스장 운동 시 발의 피로를 최소화해주는 운동화. {'name': '편안한 러닝화', 'category': '패션'} 0.5894138813018799


### 그래프 DB를 이용한 신뢰도 분석
- 제품검색은 VectorDB가 잘함
- 이 제품을 파는 판매자나 리뷰가 믿을만한가? ->관계데이터인 그래프DB가 유리
- 유저-상ㅍ무간의 구매 및 리뷰 그래프 모델링

In [ ]:
# 그래프 생성
G = nx.DiGraph()
# 상품 노드추가(VectorDB id 사용)
for p in products:
    G.add_node( p['id'], type='Product' , name = p['name']  )

# 유저노드 추가
users = ['User_A','User_B','User_C',"Spammer_1","Spammer_2"]
for u in users:
    G.add_node(u, type='User', name=u)
# 구매 및 추천 관계 추가
# 정상유저들의 패턴 : 여러 상품을 구매하고 교류
edges = [
    ("User_A", "p5", "PURCHASED"), ("User_A", "p4", "PURCHASED"),
    ("User_B", "p5", "PURCHASED"), ("User_B", "p1", "PURCHASED"),
    ("User_C", "p1", "PURCHASED"), ("User_C", "p3", "PURCHASED"),
    ("User_A", "User_B", "FOLLOWS"), ("User_B", "User_C", "FOLLOWS")
]
# 사기패턴 : 특정 스패머들이 특정 상품(p2)만 집중적으로 가짜 구매/리뷰
spam_edges = [
    ("Spammer_1", "p2", "PURCHASED"), ("Spammer_2", "p2", "PURCHASED"),
    ("Spammer_1", "p2", "REVIEWED_5_STAR"), ("Spammer_2", "p2", "REVIEWED_5_STAR"),
    ("Spammer_1", "Spammer_2", "FOLLOWS") # 자기들끼리만 연결됨
]

p1
p2
p3
p4
p5
